# Does the sign-up promotion change annual spend?

This notebook walks through the analysis of the promotion in `train.csv`: what the raw comparison says, why it cannot be taken at face value, what adjusted estimators give, how strong hidden confounding would have to be to change the conclusion, whether any group of members responds, and how a randomised test should be designed to settle the question. Methods and references are in `docs/methodology.md`, sections 10 and 10c.

In [1]:
import warnings

import pandas as pd

from loyalty_spend.causal.analysis import analyse_promotion
from loyalty_spend.causal.data import PRE_TREATMENT, from_frame
from loyalty_spend.causal.targeting import analyse_targeting
from loyalty_spend.data.io import load_clean

warnings.filterwarnings("ignore")
pd.set_option("display.precision", 3)
train, _ = load_clean("../data/raw/train.csv", impute=False)
train.shape

(996, 8)

## The raw comparison

Members who joined through the promotion spend about one unit less, but spend varies by about 27 between members, so a difference of one is well within noise. The arms are similar in tenure, and differ somewhat in joining month (the July share differs by 0.2 standard deviations).

In [2]:
train.groupby("promotion", observed=True)["spend"].agg(["count", "mean", "std"])

,count,mean,std
promotion,,,
Yes,494,128.786,26.869
No,502,129.780,26.795


In [3]:
pd.crosstab(train["loyalty_years"], train["promotion"], normalize="columns")

promotion,Yes,No
loyalty_years,,
0-1,0.320,0.313
1-3,0.182,0.177
3-5,0.196,0.219
5-10,0.172,0.175
10+,0.130,0.116


## Adjusted estimates

Only attributes fixed at sign-up are adjusted for (region, joining month and tenure band); first-month spend is measured after joining and could itself be affected by the promotion. Regression adjustment, inverse probability weighting, augmented IPW and double machine learning should agree if the models are adequate.

In [4]:
data = from_frame(train)
result = analyse_promotion(data, seed=42)
result.estimates[["method", "estimand", "estimate", "se", "ci_lower", "ci_upper"]]

,method,estimand,estimate,se,ci_lower,ci_upper
0,Difference in means,ate,-0.994,1.700,-4.326,2.339
1,Regression adjustment,ate,-0.017,0.020,-0.055,0.020
2,IPW,ate,0.124,1.616,-2.995,3.553
3,AIPW,ate,-0.015,0.021,-0.056,0.026
4,DML (partially linear),ate,0.009,0.064,-0.116,0.134
5,Regression adjustment,att,-0.018,0.021,-0.058,0.022
6,IPW,att,0.059,1.686,-3.263,3.550
7,AIPW,att,-0.016,0.023,-0.060,0.028


The difference in means has a standard error of 1.7. Adjusting for tenure, month and region removes almost all of the variation in spend, so the standard error falls to about 0.02 and the estimate to within a few hundredths of zero. The IPW estimates are as noisy as the raw comparison, because weighting balances the attributes without using how strongly they predict spend; the augmented and regression estimators use both.

## Overlap and balance

Weighting estimators need every kind of member to have a real chance of either arm. Propensity scores between about 0.26 and 0.69 and weighted standardized differences near zero show that this holds.

In [5]:
pd.Series(result.overlap)

treated_min        0.264
treated_max        0.693
control_min        0.261
control_max        0.692
share_extreme      0.000
ess_treated      480.824
ess_control      490.339
n_treated        494.000
n_control        502.000
dtype: float64

In [6]:
result.balance.sort_values("smd_weighted", key=abs, ascending=False).head(5)

,covariate,smd_unweighted,smd_weighted,balanced
9,joining_month=Jul,-0.198,-0.011,True
20,loyalty_years=5-10,-0.009,0.008,True
7,joining_month=Feb,0.084,0.008,True
17,loyalty_years=1-3,0.013,-0.007,True
8,joining_month=Jan,0.018,0.007,True


## Sensitivity to hidden confounding

The robustness value is the share of residual variance that an omitted confounder would need to explain, in both the treatment and the outcome, to move the estimate to zero. A value near 0.03 is small, but here the estimate is already zero: the question is what effects can be ruled out, and the interval excludes effects larger than about 0.06 in either direction.

In [7]:
pd.Series(result.sensitivity)

estimate     -1.686e-02
se            1.963e-02
t_stat       -8.588e-01
partial_r2    7.551e-04
rv            2.711e-02
rv_alpha      0.000e+00
dtype: float64

## Does the promotion work for anyone?

A doubly robust learner estimates conditional effects with cross-fitting, and the best linear predictor tests whether those predictions carry real heterogeneity. Repeated sample splitting avoids conclusions that depend on one split.

In [8]:
targeting = analyse_targeting(data, "dr", seed=42)
pd.Series(targeting.blp)

ate                     -0.007
ate_se                   0.029
heterogeneity           -0.058
heterogeneity_se         0.164
heterogeneity_p_value    1.000
dtype: float64

In [9]:
targeting.gates

,group,n,predicted,estimated,se
0,1,99.0,0.275,-0.019,0.077
1,2,100.0,0.088,-0.051,0.052
2,3,99.0,-0.012,0.010,0.059
3,4,100.0,-0.118,0.004,0.061
4,5,100.0,-0.314,0.017,0.070


The heterogeneity loading is not distinguishable from zero and the group effects overlap: there is no subgroup for which offering the promotion would pay.

## Settling the question with a randomised test

The observational conclusion rests on unconfoundedness. A test that randomises the offer within strata of tenure band and region removes that assumption. Because tenure, joining month and region explain almost all of the variance in spend, an adjusted analysis needs very few members.

In [10]:
from loyalty_spend.experiment.power import sample_size
from loyalty_spend.experiment.sequential import boundaries, design_properties

sd = train["spend"].std()
for r2, k, label in [(0.0, 0, "difference in means"), (0.99987, 24, "adjusted for all attributes")]:
    print(f"{label:30s} members per arm for an effect of 0.1: {sample_size(0.1, sd, r2=r2, n_covariates=k)[1]:,}")

difference in means            members per arm for an effect of 0.1: 1,129,414
adjusted for all attributes    members per arm for an effect of 0.1: 148


In [11]:
looks = [0.25, 0.5, 0.75, 1.0]
bounds = boundaries(looks)
props = design_properties(looks, bounds, drift=2.8016)
pd.DataFrame({"information": looks, "boundary": bounds, "stop_probability": props.stop_probabilities})

,information,boundary,stop_probability
0,0.25,3.920,0.006
1,0.50,2.774,0.208
2,0.75,2.298,0.343
3,1.00,2.043,0.230


With four looks and O'Brien-Fleming spending, the first look needs a z-statistic of 3.9, and the design keeps most of the power of a fixed test while stopping before the last look in more than half of the trials when the effect is real. `loyalty-spend experiment` adds the simulated operating characteristics on resampled members.

## Conclusion

The raw difference of one unit is noise. Adjusted for what is known at sign-up, the effect is zero within about 0.04, no subgroup responds, and a stratified randomised test with regression adjustment could confirm this with a few hundred members.